# Gradient checkpointing in practice: profiling a nanoGPT-style Transformer

Walkthrough for the gradient-checkpointing file ([`../04_gradient_checkpointing.md`](../04_gradient_checkpointing.md)) — its exercise, expanded into runnable code. We train a small GPT (6 layers, `D=384`, `S=512`, batch 16) three ways and watch the memory:

1. **Vanilla training** — measure peak memory and the activation memory autograd keeps alive.
2. **Per-block gradient checkpointing** — same measurement. Expect a large drop in cached-activation memory (the headline number depends on what fraction of memory is activations vs. weights), at ~1.3× step time.
3. **Spend the freed memory on batch size** — push `B` up until we're back at the vanilla memory budget. Per-step time grows with `B`, but tokens/sec holds or improves while each step processes far more samples. This is the practical win.

We also verify the correctness claim from the main file's self-check #3: checkpointed gradients are identical to vanilla gradients (up to kernel nondeterminism).

**Hardware notes.** Written for a CUDA GPU (a few GB free is plenty). It degrades gracefully: on Apple Silicon (`mps`) everything runs but PyTorch exposes no *peak*-memory counter, so we rely on the cached-activation probe; on CPU it runs slowly and memory numbers are skipped. The relative effects (activation drop, ~1.3× step time) show up everywhere.


In [ ]:
import math, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"device: {device} | torch {torch.__version__}")

class Config:
    n_layer    = 6
    n_head     = 6
    d_model    = 384          # D
    block_size = 512          # S
    vocab_size = 65           # char-level; reset from data below
    use_sdpa   = False        # False = materialize the (B,H,S,S) attention matrix (see note below)

cfg = Config()
BATCH = 16


## The model

A nanoGPT-style decoder: token + learned position embeddings, pre-norm blocks (`x + attn(ln(x))`, `x + mlp(ln(x))`), final LayerNorm, weight-tied LM head, GELU FFN with the usual `4D` expansion. Init is nanoGPT's `N(0, 0.02)` — without it, the tied embedding/head starts at the `nn.Embedding` default `N(0,1)` and the initial loss is garbage instead of `ln(V) ≈ 4.17`.

Two deliberate choices, both about making the memory experiment legible:

- **Attention materializes the `(B, H, S, S)` matrix** (`use_sdpa = False`). This is the textbook implementation the main file's `k ≈ 10–20` accounting assumes, and those quadratic tensors are exactly the memory that checkpointing frees. With `F.scaled_dot_product_attention`, FlashAttention never materializes the matrix at all — the "selective recomputation" section of the main file — which shrinks vanilla activation memory before checkpointing gets a chance to. Rerun the notebook with `use_sdpa = True` afterward and compare; it's the most instructive single toggle here.
- **Char-level vocab (`V = 65`)** keeps the logits tensor `(B, S, V)` tiny, so the profile is dominated by the blocks — the thing checkpointing acts on — not by the LM head.

The checkpointing switch is the loop in `GPT.forward`: per block, either call it directly or route it through `torch.utils.checkpoint.checkpoint`. That one branch is the entire intervention — it's also essentially what HuggingFace's `gradient_checkpointing_enable()` does under the hood. With `use_reentrant=False` (the modern mode), the wrapped function's forward runs under `no_grad` semantics — nothing inside is cached — and only the block *input* survives. On backward, the block is re-run with grad tracking to rebuild its internals just-in-time.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.d_head = cfg.d_model // cfg.n_head
        self.qkv  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.use_sdpa = cfg.use_sdpa
        self.register_buffer("mask", torch.tril(
            torch.ones(cfg.block_size, cfg.block_size, dtype=torch.bool)
        ).view(1, 1, cfg.block_size, cfg.block_size))

    def forward(self, x):
        B, S, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q = q.view(B, S, self.n_head, self.d_head).transpose(1, 2)   # (B, H, S, D_h)
        k = k.view(B, S, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, S, self.n_head, self.d_head).transpose(1, 2)
        if self.use_sdpa:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, H, S, S)  <- the big one
            att = att.masked_fill(~self.mask[:, :, :S, :S], float("-inf"))
            att = F.softmax(att, dim=-1)                              # (B, H, S, S)  saved for backward
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, S, D)
        return self.proj(y)

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc   = nn.Linear(cfg.d_model, 4 * cfg.d_model, bias=False)
        self.proj = nn.Linear(4 * cfg.d_model, cfg.d_model, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))                          # (B, S, 4D) intermediates

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(cfg.d_model), CausalSelfAttention(cfg)
        self.ln2, self.mlp  = nn.LayerNorm(cfg.d_model), MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.blocks  = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layer))
        self.ln_f    = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight   # weight tying
        self.checkpoint_blocks = False              # the switch this notebook is about
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets):
        B, S = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(S, device=idx.device))
        for block in self.blocks:
            if self.checkpoint_blocks and self.training:
                x = checkpoint(block, x, use_reentrant=False)   # save only the block input
            else:
                x = block(x)                                    # save everything inside
        logits = self.lm_head(self.ln_f(x))
        return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))


## Data

Tiny Shakespeare, char-level — the classic nanoGPT starter corpus — with a random-tokens fallback if there's no network. For *memory and timing*, data content is irrelevant (shapes are shapes); real text just makes the loss curve meaningful.


In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    text = urllib.request.urlopen(url, timeout=10).read().decode()
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    cfg.vocab_size = len(chars)
    print(f"tiny shakespeare: {len(data):,} tokens, vocab {cfg.vocab_size}")
except Exception as e:
    print(f"download failed ({e}); using random tokens — fine for profiling")
    data = torch.randint(0, cfg.vocab_size, (1_000_000,))

def get_batch(batch_size):
    ix = torch.randint(len(data) - cfg.block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + cfg.block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + 1 + cfg.block_size] for i in ix])
    return x.to(device), y.to(device)


## How we measure

Three numbers per configuration, after warmup steps so gradients and AdamW states already exist:

- **static** — bytes allocated *between* steps: parameters + gradients + optimizer states (+ buffers). Checkpointing can't touch this; only things like ZeRO/FSDP or optimizer choice can.
- **activations** — bytes allocated at the *end of forward, before backward*, minus static. This is precisely what autograd is keeping alive for backward — the quantity checkpointing attacks. (Measuring it as a forward-end delta works on any backend with an allocator query, including `mps`.)
- **peak** — CUDA's high-water mark across the whole step (`max_memory_allocated`). This is what actually OOMs you. Note peak > static + activations: backward transiently holds gradient buffers next to activations, and under checkpointing the recompute working set spikes here.

Timing: synchronize before/after (GPU work is async — without `synchronize()` you time the kernel *launches*, not the kernels), average over several steps.

One pitfall this avoids: `zero_grad(set_to_none=False)` keeps gradient buffers allocated between steps so "static" is stable. The default `set_to_none=True` frees them, which would smear gradient memory into the per-step delta and muddy the comparison.


In [ ]:
def sync():
    if device == "cuda": torch.cuda.synchronize()
    elif device == "mps": torch.mps.synchronize()

def mem_allocated():
    if device == "cuda": return torch.cuda.memory_allocated()
    if device == "mps": return torch.mps.current_allocated_memory()
    return 0

def reset_peak():
    if device == "cuda": torch.cuda.reset_peak_memory_stats()

def mem_peak():
    return torch.cuda.max_memory_allocated() if device == "cuda" else None

MiB = 2**20
def fmt(b): return f"{b / MiB:8.1f} MiB" if b is not None else "     n/a"

def profile_training(model, batch_size, n_steps=10, n_warmup=3):
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    model.train()
    for _ in range(n_warmup):                      # materialize grads + AdamW states
        model(*get_batch(batch_size)).backward()
        opt.step(); opt.zero_grad(set_to_none=False)
    sync(); reset_peak()
    static = mem_allocated()
    act_bytes = None
    t0 = time.perf_counter()
    for i in range(n_steps):
        loss = model(*get_batch(batch_size))
        if i == 0:
            sync()
            act_bytes = mem_allocated() - static   # what autograd cached for backward
        loss.backward()
        opt.step(); opt.zero_grad(set_to_none=False)
    sync()
    return dict(static=static, activations=act_bytes, peak=mem_peak(),
                step_time=(time.perf_counter() - t0) / n_steps, loss=loss.item())

def report(name, r, batch_size):
    toks = batch_size * cfg.block_size / r["step_time"]
    print(f"{name:22s} static {fmt(r['static'])} | activations {fmt(r['activations'])} | "
          f"peak {fmt(r['peak'])} | {r['step_time']*1e3:7.1f} ms/step | {toks/1e3:7.1f}k tok/s | "
          f"loss {r['loss']:.3f}")


## Experiment 1 — vanilla training

**Prediction first** (fp32, `B=16, S=512, D=384, H=6, L=6`):

- One residual-stream tensor `B·S·D·4` ≈ **12.6 MB**. The attention matrix `B·H·S·S·4` ≈ **101 MB** per layer — eight residual streams' worth, from one tensor. The FFN intermediates `B·S·4D·4` ≈ 50 MB each.
- Per block, autograd saves the softmax output (~101 MB), two FFN intermediates (~101 MB), and roughly 6–8 `B·S·D`-sized tensors (LayerNorm inputs, Q/K/V, attention output, projection inputs — ~90 MB). Call it **~290 MB per block, ~1.7–2 GB across 6 blocks**. That's the `k ≈ 10–20` factor from the main file, with attention's quadratic term doing most of the damage even at `S=512`.
- Static: ~10.8 M params ≈ 43 MB, gradients 43 MB, AdamW `m` and `v` 86 MB → **~170 MB**. Activations should dwarf it by ~10× — the main file's point that activations, not parameters, are what limit training.


In [ ]:
torch.manual_seed(1337)
model = GPT(cfg).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

model.checkpoint_blocks = False
vanilla = profile_training(model, BATCH)
report("vanilla", vanilla, BATCH)


## Experiment 2 — checkpoint every block

**Prediction.** At the end of forward, the only survivors from the blocks are the six **block inputs** (the checkpoints): `6 × 12.6 MB ≈ 76 MB`, plus embedding/head leftovers. So the *activations* number should collapse from ~1.8 GB to ~0.1 GB — roughly **15–20×** on this measure, because the boundary tensor is so much smaller than the ~290 MB living inside each block (the "why every-block works for Transformers" argument: boundary ≪ internals).

The *peak* falls less. During backward, recomputing a block transiently rebuilds its full ~290 MB working set, and backward's own buffers ride on top of static memory. Peak ≈ static + checkpoints + one block's working set + backward scratch — expect the **4–6×** headline range on peak rather than 15×. The recompute buffer is exactly term 2 in the main file's `L/N + N` analysis: you never get to keep *both* terms small.

**Step time:** forward now runs every block twice (once discarded, once during backward), so a `3F` step becomes `4F` → predicted **~1.33×** slower.


In [ ]:
model.checkpoint_blocks = True
ckpt = profile_training(model, BATCH)

report("vanilla", vanilla, BATCH)
report("checkpointed", ckpt, BATCH)

if vanilla["activations"] and ckpt["activations"]:
    print(f"\ncached-activation reduction: {vanilla['activations'] / ckpt['activations']:.1f}x")
if vanilla["peak"] and ckpt["peak"]:
    print(f"peak-memory reduction:       {vanilla['peak'] / ckpt['peak']:.1f}x")
print(f"step-time ratio:             {ckpt['step_time'] / vanilla['step_time']:.2f}x")


### What just happened during backward

Walk one step of the checkpointed run:

1. **Forward**: block 1 runs under no-grad semantics — its softmax output, FFN intermediates, Q/K/V all get freed as soon as the block output exists. Only the block's *input* tensor is kept. Same for blocks 2–6. Forward ends holding six 12.6 MB tensors instead of six ~290 MB working sets.
2. **Backward reaches block 6**: autograd hits the checkpoint node, which says "I don't have block 6's internals — recompute them." It re-runs block 6's forward *from the saved input*, this time with grad tracking, materializing the ~290 MB working set. Backward then proceeds through it normally and frees it.
3. Repeat for blocks 5, 4, … 1. At any moment, at most **one** block's internals exist — that's the transient spike that sets the checkpointed peak.

The step-time ratio lands near 1.33× rather than exactly: the LM head, embeddings, optimizer, and data loading aren't recomputed (they dilute the overhead), while recompute kernels may run at different efficiency than the original forward (they can add to it).


## Experiment 3 — spend the savings: scale the batch

Memory saved is only useful if you buy something with it. The standard purchase is **batch size** (or sequence length — same currency). We grow `B` with checkpointing on, until peak memory is back at the vanilla budget.

**Prediction:** per-step time grows roughly linearly with `B`, but **tokens/sec holds or improves** — bigger batches keep the GPU's compute units busier per kernel launch, which usually more than repays the 1.33× recompute tax. The win is *throughput at fixed memory*: each optimizer step now consumes several times more samples on the same card. (Whether bigger batches help *optimization* per token is a separate question — Part 2.4's territory.)


In [ ]:
budget = vanilla["peak"] or (vanilla["static"] + (vanilla["activations"] or 0))
print(f"memory budget (vanilla peak): {fmt(budget)}\n")
report("ckpt", ckpt, BATCH)

for B in [32, 64, 96, 128]:
    try:
        r = profile_training(model, B, n_steps=5)
    except torch.OutOfMemoryError:
        print(f"B={B}: OOM — past the budget"); break
    report(f"ckpt", r, B)
    used = r["peak"] or (r["static"] + (r["activations"] or 0))
    if used > budget:
        print(f"\nB={B} exceeds the vanilla budget — the previous row is the operating point.")
        break


## Correctness: checkpointing does not change the gradients

Self-check #3 of the main file: recomputation is exact, so gradients should match vanilla. `checkpoint` snapshots and restores RNG state for the wrapped function, so even dropout (we have none) would replay identically. We build the same model twice from the same seed, feed the same batch, and compare every parameter gradient.

Expect **zero or ~1e-8** difference: bitwise-identical when the backend's kernels are deterministic; on `mps` and some CUDA kernel choices, floating-point reduction order varies between the original and recomputed forward, leaving noise at the last-ulp level. Either way: checkpointing costs compute, not correctness.


In [ ]:
def grads_for(use_ckpt):
    torch.manual_seed(123)
    m = GPT(cfg).to(device)
    m.checkpoint_blocks = use_ckpt
    m.train()
    torch.manual_seed(7)
    m(*get_batch(4)).backward()
    return [p.grad.clone() for p in m.parameters()]

max_diff = max((a - b).abs().max().item() for a, b in zip(grads_for(False), grads_for(True)))
print(f"max |grad_vanilla - grad_checkpointed| over all params: {max_diff:.2e}")


## Takeaways

- **Activations are the budget.** Static memory (params + grads + AdamW) was a fraction of the activation memory even for this 10.8M-param toy; at `S=8192` and 7B params the imbalance is far worse (quadratic attention term, more layers, bigger `D`).
- **Checkpointing trades one number for another.** Cached activations drop to the block-boundary tensors; peak drops by the headline 4–6×; step time pays ~1.33× (`3F → 4F`). The peak can't drop further than the recompute working set — the `L/N + N` tension from the main file, in the flesh.
- **The practical win is throughput at fixed memory**: refill the freed memory with batch (or context length) and you process several times the samples per step on the same GPU at similar tokens/sec. This is why it's always-on at scale.
- **Now rerun with `use_sdpa = True`.** FlashAttention removes the `(B,H,S,S)` tensors from existence rather than recomputing them via checkpointing — vanilla activation memory shrinks substantially before checkpointing does anything, and the checkpointing ratio drops accordingly. Selective recomputation, implemented in the kernel. (Part 7.2.)
- **Compose with mixed precision** (file [`../../2.4_optimization/05_mixed_precision.md`](../../2.4_optimization/05_mixed_precision.md)): bf16 halves bytes-per-activation, checkpointing cuts the count — the savings multiply.
